# Learning Objectives

* Build a real, per-sample Pegasus workflow from a set of Python wrapper scripts and an Apptainer container.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the scientific outputs (trajectory files, a methods report, and an RMSD plot) produced by a run.

# GROMACS Molecular Dynamics Workflow

This workflow runs a per-sample GROMACS molecular dynamics (MD) simulation pipeline, converted from the
[nf-core/moleculardynamics](https://github.com/nf-core/moleculardynamics) Nextflow pipeline. Each sample
in the samplesheet — a structure file (PDB) plus four GROMACS `.mdp` parameter files — runs an independent,
linear 10-step pipeline. Samples have no cross-sample dependencies, so Pegasus runs them all in parallel.
See `README.md` in this repository for the full pipeline description, output listing, and conversion notes.

**Workflow Jobs (per sample):**
1. `pdb_clean_and_check_missing_atoms` - strip HETATM/CONECT records from the input PDB, then fail fast if it has missing atoms
2. `topology` - `gmx pdb2gmx`: PDB -> topology (.gro/.top/.itp)
3. `solvation` - `gmx editconf`/`solvate`/`grompp`/`genion`: box + solvent + ions
4. `energy_min` - `gmx grompp`/`mdrun`: energy minimization
5. `nvt_equilibration` - `gmx grompp`/`mdrun`: NVT equilibration
6. `npt_equilibration` - `gmx grompp`/`mdrun`: NPT equilibration
7. `production` - `gmx grompp`/`mdrun`/`report-methods`: production MD run
8. `post_processing` - `gmx trjconv`: remove periodicity (PBC) artifacts
9. `analysis_rmsd` - `gmx rms`: RMSD of the trajectory
10. `analysis_plot` - `matplotlib`: plot the RMSD `.xvg` to a PNG

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

`prepare_test_data.sh` downloads a small real test case — hen egg-white lysozyme (PDB
[1AKI](https://www.rcsb.org/structure/1AKI)), the classic GROMACS protein-in-water tutorial system — into
`data/1AKI/`, writes four small `.mdp` parameter files (deliberately tiny step counts, for a fast demo run,
**not** suitable for production MD), and writes `data/samplesheet.csv` pointing at them.

Run this on its own any time you just want fresh input data and a matching samplesheet; it's independent of
the rest of this notebook.

In [ ]:
!./prepare_test_data.sh

## 2. Build the Apptainer Container

The workflow's jobs run inside a single Apptainer container (`Apptainer/Gromacs_Container.def`) that bundles
GROMACS, Python 3, and matplotlib. Building it can take a few minutes; skip the rebuild if the `.sif` already
exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Gromacs_Container.sif ]; then
    apptainer build Apptainer/Gromacs_Container.sif Apptainer/Gromacs_Container.def
else
    echo "Container already built: Apptainer/Gromacs_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `GromacsMDWorkflow` class from `workflow_generator.py` — the same generator used
by the command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs)
from `data/samplesheet.csv`, and writes it to `workflow.yml`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import GromacsMDWorkflow, parse_samplesheet

SAMPLESHEET = "data/samplesheet.csv"
DAGFILE = "workflow.yml"
EXEC_SITE = "compute"

samples = parse_samplesheet(SAMPLESHEET)
print(f"Samples: {[s['sample'] for s in samples]}")

workflow = GromacsMDWorkflow(samples=samples, gmx_cmd="gmx", dagfile=DAGFILE)

print("Creating workflow properties...")
workflow.create_pegasus_properties()

print("Creating execution sites...")
workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating GROMACS MD workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nGROMACS MD Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. For each sample, the graph shows the
linear 10-job pipeline described above; independent samples appear as separate parallel chains.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` HTCondor site defined in the site
catalog created above.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`, per sample:

| Output | Description |
|--------|-------------|
| `{sample}_md.gro`, `{sample}_md.tpr`, `{sample}_md.xtc` | Production MD trajectory + structure |
| `{sample}_MD_REPORT.out` | Simulation methods report (`gmx report-methods`) |
| `{sample}_rmsd.xvg` | RMSD of the trajectory vs. time |
| `{sample}_rmsd.png` | RMSD vs. time plot |

In [ ]:
!ls -ltR output/

Let's look at the RMSD plot for our sample — the protein backbone's structural deviation over
the course of the simulation:

In [ ]:
SAMPLE = samples[0]["sample"]

Image(filename=f"output/{SAMPLE}_rmsd.png")

And the simulation's methods report:

In [ ]:
!cat output/{SAMPLE}_MD_REPORT.out